In [1]:
import pandas as pd

In [15]:
# Chargement
 
df_prod  = pd.read_parquet("../dataset_1.parquet", engine="fastparquet")
df_sites = pd.read_parquet("../dataset_2.parquet", engine="fastparquet")
df_meteo = pd.read_parquet("../dataset_3.parquet", engine="fastparquet")

In [16]:
df_prod.head()

,site_name,delivery_time,production,installed_capacity
0,Nobelwind Offshore Windpark,2022-12-31 23:00:00+00:00,162.7175,165.0
1,Nobelwind Offshore Windpark,2023-01-01 00:00:00+00:00,162.6850,165.0
2,Nobelwind Offshore Windpark,2023-01-01 01:00:00+00:00,162.6800,165.0
3,Nobelwind Offshore Windpark,2023-01-01 02:00:00+00:00,162.7300,165.0
4,Nobelwind Offshore Windpark,2023-01-01 03:00:00+00:00,162.7125,165.0


In [18]:
df_prod['site_name'].unique()

array(['Nobelwind Offshore Windpark', 'Rentel Offshore WP',
       'Norther Offshore WP', 'Northwester 2', 'Mermaid Offshore WP',
       'Seastar Offshore WP', 'Belwind Phase 1', 'Northwind',
       'Thorntonbank - C-Power - Area NE',
       'Thorntonbank - C-Power - Area SW'], dtype=object)

In [17]:
df_sites.head()

,site_name,latitude,longitude
0,Nobelwind Offshore Windpark,51.6631,2.8339
1,Rentel Offshore WP,51.5910,2.9440
2,Norther Offshore WP,51.5280,3.0320
3,Northwester 2,51.6875,2.7488
4,Mermaid Offshore WP,51.6300,2.8597


In [19]:
df_sites['site_name'].unique()

array(['Nobelwind Offshore Windpark', 'Rentel Offshore WP',
       'Norther Offshore WP', 'Northwester 2', 'Mermaid Offshore WP',
       'Seastar Offshore WP', 'Belwind Phase 1', 'Northwind',
       'Thorntonbank - C-Power - Area NE',
       'Thorntonbank - C-Power - Area SW'], dtype=object)

In [20]:
df_meteo.head()

,site_name,delivery_time,wind_speed_10m,wind_speed_100m,wind_direction_10m,wind_direction_100m,wind_gusts_10m,temperature_2m,dewpoint_2m,apparent_temperature,...,snowfall,cloud_cover,cloud_cover_low,cloud_cover_mid,cloud_cover_high,shortwave_radiation,direct_radiation,diffuse_radiation,weather_code,sunshine_duration
0,Belwind Phase 1,2023-01-01 00:00:00+00:00,14.603082,19.897738,218.04709,219.28940,20.7,12.25,8.85,4.282408,...,0.0,100.0,53.0,100.0,98.0,0.0,0.0,0.0,51.0,0.0
1,Belwind Phase 1,2023-01-01 01:00:00+00:00,16.182089,21.681328,215.94937,217.50421,20.8,12.10,8.80,3.290131,...,0.0,100.0,18.0,100.0,100.0,0.0,0.0,0.0,51.0,0.0
2,Belwind Phase 1,2023-01-01 02:00:00+00:00,17.969420,23.809662,226.80397,228.74626,24.1,11.85,9.50,2.291797,...,0.0,100.0,31.0,100.0,100.0,0.0,0.0,0.0,51.0,0.0
3,Belwind Phase 1,2023-01-01 03:00:00+00:00,14.792228,19.860010,227.46579,229.49266,23.9,11.80,9.85,4.007824,...,0.0,100.0,27.0,24.0,100.0,0.0,0.0,0.0,3.0,0.0
4,Belwind Phase 1,2023-01-01 04:00:00+00:00,15.001333,19.915070,227.16109,228.86830,19.7,11.75,9.30,3.694952,...,0.0,100.0,3.0,45.0,100.0,0.0,0.0,0.0,3.0,0.0


In [21]:
df_meteo['site_name'].unique()

array(['Belwind Phase 1', 'Mermaid Offshore WP',
       'Nobelwind Offshore Windpark', 'Norther Offshore WP',
       'Northwester 2', 'Northwind', 'Rentel Offshore WP',
       'Seastar Offshore WP', 'Thorntonbank - C-Power - Area NE',
       'Thorntonbank - C-Power - Area SW'], dtype=object)

In [26]:
# Timestamps en UTC timezone-aware
df_prod["delivery_time"] = pd.to_datetime(df_prod["delivery_time"], utc=True)
df_meteo["delivery_time"] = pd.to_datetime(df_meteo["delivery_time"], utc=True)

In [27]:
# Ajout des données météo au dataset de production
df = df_prod.merge(
    df_meteo,
    on=["site_name", "delivery_time"],
    how="inner"   # garde la période commune aux deux datasets
)

In [28]:
# Ajout des coordonnées GPS
df = df.merge(df_sites, on="site_name", how="left")

In [29]:
# Tri par site et par date
df = df.sort_values(["site_name", "delivery_time"]).reset_index(drop=True)

In [30]:
print("=== Dataset final ===")
print(f"Lignes   : {len(df):,}")
print(f"Colonnes : {df.shape[1]}")
print(f"Parcs    : {df['site_name'].nunique()}  →  {sorted(df['site_name'].unique())}")
print(f"Période  : {df['delivery_time'].min()} → {df['delivery_time'].max()}")
print(f"\nColonnes :\n{df.dtypes.to_string()}")
print(f"\nValeurs manquantes :\n{df.isnull().sum()[df.isnull().sum() > 0].to_string() or 'aucune'}")

=== Dataset final ===
Lignes   : 274,790
Colonnes : 27
Parcs    : 10  →  ['Belwind Phase 1', 'Mermaid Offshore WP', 'Nobelwind Offshore Windpark', 'Norther Offshore WP', 'Northwester 2', 'Northwind', 'Rentel Offshore WP', 'Seastar Offshore WP', 'Thorntonbank - C-Power - Area NE', 'Thorntonbank - C-Power - Area SW']
Période  : 2023-01-01 00:00:00+00:00 → 2026-02-18 22:00:00+00:00

Colonnes :
site_name                            object
delivery_time           datetime64[ns, UTC]
production                          float64
installed_capacity                  float64
wind_speed_10m                      float64
wind_speed_100m                     float64
wind_direction_10m                  float64
wind_direction_100m                 float64
wind_gusts_10m                      float64
temperature_2m                      float64
dewpoint_2m                         float64
apparent_temperature                float64
pressure_msl                        float64
surface_pressure                  

In [33]:
missing = df[df["production"].isna()]

print(missing.groupby("site_name").size())       
print(missing["delivery_time"].dt.year.value_counts()) 
print(missing["delivery_time"].diff().dt.total_seconds().div(3600))

site_name
Norther Offshore WP                 24
Northwind                           24
Rentel Offshore WP                  24
Thorntonbank - C-Power - Area NE    24
dtype: int64
delivery_time
2024    48
2025    48
Name: count, dtype: int64
94579     NaN
94580     1.0
94581     1.0
94582     1.0
94583     1.0
         ... 
240035    1.0
240036    1.0
240037    1.0
240038    1.0
240039    1.0
Name: delivery_time, Length: 96, dtype: float64


très probablement des arrêts de maintenance planifiée ou des pannes --> remplacer avec 0

In [34]:
df["production"] = df["production"].fillna(0)
assert df["production"].isna().sum() == 0

In [35]:
df.to_parquet("dataset_final.parquet", engine="fastparquet", index=False)